# generation of a json with the values: Automatique verification of the constraints


In [3]:
import pandas as pd
import json
from pathlib import Path
import os
from enum import Enum
from pydantic import BaseModel, constr
from openai import OpenAI
from scipy.special import softmax
from datetime import datetime

In [1]:
# from dotenv import load_dotenv
# import os

# load_dotenv()

In [3]:
client = OpenAI(
)
model_id= "gpt-5.1"
llm_used= model_id

In [5]:
df_generation= pd.read_json("../TextGeneration/Ouptut/outputgreedy_google_medgemma-27b-text-itTrain.json")

keys_to_remove = ["original_index","t", "n", "m"]
# Remove keys from each dictionary in the 'data' column
df_generation['entry_data'] = df_generation['entry_data'].apply(lambda x: {k: v for k, v in x.items() if k not in keys_to_remove})


In [21]:
def format_text_att(att,expected_value):
    attribute=dict_att_names[att]
    if expected_value=="unknown":
        return f"unknown"
    
    att_expected_value=f"{expected_value}"
    if att=='embols_vasculaires_tumor_0': 
        if expected_value=="0" or expected_value==0:
            att_expected_value="non"
        elif expected_value=="1" or expected_value==1:
            att_expected_value="oui"
        else: 
            att_expected_value="non évalué"
    elif att=="taille_tumor_0":
        att_expected_value=f"{expected_value} mm (peut être exprimée en cm, mais les valeurs doivent être respectées)" 
    elif att=="ki67_tumor_0":
        att_expected_value=f"{expected_value} %" 
    elif att == "ganglions_preleves":
        att_expected_value=f"{expected_value}"
    elif att == "ganglions_atteints":
        att_expected_value=f"{expected_value}"
    elif att == "ref_grade_tumor_0":
        if expected_value== 'sbr1':
            att_expected_value="I"
        if expected_value == 'sbr2':
            att_expected_value="II"
        if expected_value == 'sbr3':         
            att_expected_value="III"
        if expected_value== 'SBR1':
            att_expected_value="I"
        if expected_value == 'SBR2':
            att_expected_value="II"
        if expected_value == 'SBR3':         
            att_expected_value="III"
        if expected_value== 'MSBR1':
            att_expected_value="I"
        if expected_value == 'MSBR2':
            att_expected_value="II"
        if expected_value == 'MSBR3':         
            att_expected_value="III"
    # print(att, att_expected_value)

    return att_expected_value        
    

In [22]:
dict_examples_prompt={
    "type_diagnostique":"""For example, in the text: "MAMMECTOMIE DROITE AVEC CURAGE AXILLAIRE MONOBLOC", the ATTRIBUTE is mentioned,  
and we can extract the value: "mammectomie".  
Look for this information in the title.
    """,  

    "ganglions_preleves":"""For example:
- In the text: "Absence de métastase ganglionnaire (0/12)", the ATTRIBUTE is mentioned, and we can extract from "(0/12)" that the value is: 12.
- In the text: "Nombre de ganglions lymphatiques examinés : 6.", the ATTRIBUTE is mentioned, and we can extract that the value is: 6.
- In the text: "Un ganglion lymphatique sur six examinés est atteint par des métastases.", the ATTRIBUTE is mentioned, and we can extract that the value is: 6.

Note: A valid value must be explicitly provided in the text.  
If no lymph nodes were examined (e.g., "Aucun ganglion lymphatique n’a été examiné.") or if the evaluation was not performed (e.g., "Ganglions lymphatiques : non évalués."), then no value can be extracted.
    """,

    "ganglions_atteints":"""
    For example:
- In the text: "Absence de métastase ganglionnaire (0/12)", the ATTRIBUTE is mentioned, and we can extract that the value is: 0.
- In the text: "Nombre de ganglions lymphatiques métastatiques : 1.", the ATTRIBUTE is mentioned, and we can extract that the value is: 1.
- In the text: "Un ganglion lymphatique sur six examinés est atteint par des métastases.", the ATTRIBUTE is mentioned, and we can extract that the value is: 1.

Note: A valid value must be explicitly present in the text.  
If no lymph nodes were evaluated or no result is available (e.g., "Aucun ganglion lymphatique n’a été examiné.", "Ganglions lymphatiques : non évalués.", "Non analysé.", "Non réalisé."), then no value can be extracted.
    """ ,

    "rupture_capsulaire" :"""For example in the text: 'Pas de rupture capsulaire.' there is a mention to the ATTRIBUTE, and we can extract that the value is: 'non'.""",
    
    "taille_tumor_0":"""For example:
- In the text: 'Taille curanlée : 71 mm' there is a mention to the ATTRIBUTE, and we can extract that the value is: 71.
- In the text: 'Carcinome canalaire infiltrant de 15 mm de diamètre' there is a mention to the ATTRIBUTE, and we can extract that the value is: 15.
""" ,
    
    "re_tumor_0":"""For example in the text: 'Œstrogène : Estrogen Receptor (SP1) VENTANA : Réaction positive' there is a mention to the ATTRIBUTE, 
and we can extract that the value is: 'positif'.""",
    
    "rp_tumor_0":"""For example in the text: 'Progestérone : Progesteron Reccptor (1 E 2) VENTANA : Réaction négative.' there is a mention to the ATTRIBUTE, 
and we can extract that the value is: 'négatif'.""",

    "embols_vasculaires_tumor_0":"""For example in the text: "Absence d'embole vasculaire" there is a mention to the ATTRIBUTE, 
and we can extract that value is: 'non'. So: 
 - if the REQUESTED VALUE = "non", the json should be: {{"correct": "yes", "reference": "Absence d'embole vasculaire"}}
 - if the REQUESTED VALUE = "oui", the json should be: {{"correct": "no", "reference": "-"}}
The following phrases also indicate absence ("non") for “emboles vasculaires”:
    - "absence d’invasion vasculaire"
    - "absence d’invasion lympho-vasculaire"
    - "absence d’invasion lymphovasculaire”
    - "pas d’invasion lympho-vasculaire”
    - "pas d’invasion vasculaire"
    - "Aucun embole vasculaire n'est identifié"
The following phrases indicate presence ("oui"):
- “invasion vasculaire”
- “invasion lympho-vasculaire”
- “invasion lymphovasculaire”
     """,
    
    "cerb_tumor_0":"""For example in the text: 'Statut Her2Neu (anticorps prédilué Ventana) : Absence de surexpression : score 0.' there is a mention to the ATTRIBUTE, 
    and we can extract that the value is 0, which is equivalent to 'négatif'.""",
    
    "marges_saines_tumor_0":"""For example in the text: 'Coupes chiurgicales partout en tissu sain.' there is a mention to the ATTRIBUTE, 
    and we can extract that the value is 'oui'.""",
    
    "ki67_tumor_0":"""For example in the text: 'KI67 (MSKO18/Zylomed Clone K2) : 25 %.' there is a mention to the ATTRIBUTE, 
    and we can extract that the value is 25 %.""" ,
    
    "ref_morpho_name_tumor_0" :"""
For example:
- In the text: "Type histo-pathologique : carcinome canalaire infiltrant.", the ATTRIBUTE is mentioned, and we can extract the value "carcinome canalaire infiltrant", which is equivalent to "adénocarcinome canalaire infiltrant".

- In the text: "Carcinome canalaire in situ (DCIS) de 25 mm de diamètre.", the ATTRIBUTE is mentioned, and we can extract the value "carcinome canalaire in situ", which is equivalent to "carcinome intracanalaire non infiltrant".

- In the text: "Biopsie mammaire révélant un carcinome canalaire infiltrant de grade SBR I.", the ATTRIBUTE is mentioned, and we can extract the value "carcinome canalaire infiltrant", which is a subtype of breast "adénocarcinome".

Additional rules:
- Synonyms of tumor morphologies must be considered equivalent.  
  Examples:
  - "carcinome papillaire, SAI" = "carcinome papillaire infiltrant"
  - "adénocarcinome apocrine" = "carcinome infiltrant de type apocrine"

- If the ATTRIBUTE expects a **more generic morphology**, a **more specific extracted morphology** is acceptable.  
  Example:
  - The generic term "adénocarcinome" includes "carcinome canalaire infiltrant", so the extracted value is valid.
    """,
   
    "ref_grade_tumor_0" : """For example, in the text: "Grade histo-pronostique de Scarff, Bloom et Richardson modifié par Elston et Ellis (Nottingham) : 3", the ATTRIBUTE is mentioned, and we can extract that the value is: III.

Note: The notation "SBR" (modified SBR) is considered equivalent to "SBR" for grading purposes.
""",
}


def build_eval_val_prompt(att,attribute_name,val,texte):  #, att_example_output, example_textes, texte):
    if val =="unknown":
        # print(val)
        content_user = f"""
You are a helpful assistant.

You will receive:

- an ATTRIBUTE, after the token ##ATTRIBUTE##
- a French TEXT, after the token ##TEXT##

Your task is to determine whether the TEXT gives a **clear, explicit and evaluable result** for this ATTRIBUTE
(e.g. positive, negative, grade, percentage, yes, no), and if so, extract the exact phrase from the TEXT.

Important:
- The "reference" MUST be copied exactly from the TEXT:
  - It must be a contiguous substring of the TEXT.
  - You must not add, remove or change any words.
  - If you cannot find an exact substring in the TEXT, you must answer:
    {{"correct": "no", "reference": "-"}}.

{dict_examples_prompt[att]}

Decision rules:

1. If the TEXT gives a clear, explicit RESULT for the ATTRIBUTE
   (e.g. "HER2 3+", "HER2 négatif", "HER2 surexprimé" for "statut HER2"),
   answer:
   {{"correct": "yes", "reference": "exact phrase from the text giving the result"}}

2. If the ATTRIBUTE is mentioned ONLY to say it was NOT evaluated or is UNKNOWN
   (e.g. "n’a pas été évaluée", "non testé", "non recherché", "non réalisé", "non contributif",
        "résultat non interprétable", "inconnu", "pas de résultat disponible"),
   then the result is NOT known. Answer:
   {{"correct": "no", "reference": "-"}}

3. If the ATTRIBUTE is not mentioned at all in the TEXT, and no result can be deduced explicitly, answer:
   {{"correct": "no", "reference": "-"}}

4. Never infer or guess a value for the ATTRIBUTE from context or other information.
   If the explicit result is not written in the TEXT, answer:
   {{"correct": "no", "reference": "-"}}

Additional rules:
- Respond ONLY with valid JSON.
- Do NOT add comments or explanations.

Output format:
Return exactly one JSON object, for example:
{{"correct": "yes", "reference": "HER2 négatif (score 0)"}}

##ATTRIBUTE##: {attribute_name}
##TEXT##: {texte}
"""

        
    else:
        rules= """

STEP 2 — If the ATTRIBUTE is mentioned:
Extract the actual value expressed in the TEXT. 

--------------------------------------------     

STEP 3 — Compare the extracted value to the REQUESTED VALUE.
You MUST follow this rule:

- If extracted value == REQUESTED VALUE:
    Return:
    {
      "correct": "yes",
      "reference": "exact phrase from the text indicating presence or absence"
    }

- If extracted value != REQUESTED VALUE:
    Return:
    {
      "correct": "no",
      "reference": "-"
    }
IMPORTANT:  
When extracted value != REQUESTED VALUE, you MUST set "reference": "-"  
Do NOT copy any phrase from the text in this case.
                """
       
    
        if att in ["embols_vasculaires_tumor_0","marges_saines_tumor_0","rupture_capsulaire"]: 
            # boolean rules
            rules= """

STEP 2 — If the ATTRIBUTE is mentioned:
    Extract the actual value expressed in the TEXT:
    - If the text explicitly expresses presence (e.g., “présence de”, “embol(e) vasculaire identifié”), extracted value = "oui".
    - If the text expresses absence (e.g., “absence de”, “aucun”, “pas de”, “sans”), extracted value = "non".

-------------------------------------------- 

STEP 3 — Compare the extracted value to the REQUESTED VALUE.

You MUST follow this rule:

- If extracted value == REQUESTED VALUE:
    Return:
    {
      "correct": "yes",
      "reference": "exact phrase from the text indicating presence or absence"
    }

- If extracted value != REQUESTED VALUE:
    Return:
    {
      "correct": "no",
      "reference": "-"
    }

IMPORTANT:  
When extracted value != REQUESTED VALUE, you MUST set "reference": "-"  
Do NOT copy any phrase from the text in this case.
"""

    
        content_user = f"""
You are helpful assistant.

You will receive:

- an ATTRIBUTE, delimited by ##ATTRIBUTE##  
- a REQUESTED VALUE delimited by ##REQUESTED VALUE##  
- a French TEXT, delimited by ##TEXT##

Your task is to determine whether the TEXT supports the statement:
“ATTRIBUTE = REQUESTED VALUE”.

Follow these steps internally, but output ONLY the final JSON:

--------------------------------------------    

STEP 1 — Determine whether the TEXT contains any expression corresponding to the ATTRIBUTE (even if wording differs).
    If the ATTRIBUTE is not mentioned at all:
    Return ONLY this JSON:
    
    {{
      "correct": "no",
      "reference": "-"
    }}
--------------------------------------------    

{rules}

--------------------------------------------    

{dict_examples_prompt[att]}

--------------------------------------------    
Additional rules:
- Respond ONLY with a valid JSON.
- Do NOT add comments or explanations.

EXACT output format:
{{
  "correct": "yes" | "no",
  "reference": "exact phrase from the text" | "-"
}}


##ATTRIBUTE##: {attribute_name}
##REQUESTED VALUE##: {format_text_att(att,val)}
##TEXT##: {texte}

                        """
    

    
    messages = [
        {
            "role": "user",
            "content": f"{content_user}"
        }]
            
        
    # if att == "taille_tumor_0":
    #     print (messages)
    return messages



In [23]:
from typing import Literal

class ExtractionS(BaseModel):
    correct: Literal["yes", "no"]
    reference:  constr(max_length=200)



dict_att_names={"type_diagnostique":"procédure chirurgicale ou echantillon ('tumorectomie', 'mammectomie', 'biopsie', 'exérèse chirurgicale', 'curage ganglionnaire')",
            "ganglions_preleves":"nombre de ganglions lymphatiques examinés  (prélevés)",
            "ganglions_atteints":"nombre de ganglions lymphatiques metastatiques (atteintes)",
            "rupture_capsulaire" :"présence de rupture capsulaire ganglionnaire ('oui', 'non')",
            "taille_tumor_0":"""taille tumorale (mesurée en mm ou cm). 
- It refers ONLY to the SIZE OF THE TUMOR / LESION itself (tumeur, masse, lésion, carcinome, nodule, etc.).
- Do NOT treat the size of a biopsy, specimen, piece, fragment or block as tumor size.
  For example, in the sentence:
  'Réception d’une biopsie mammaire mesurant 0,5 x 0,3 x 0,2 cm.'
  there is NO information about the tumor size, only the size of the biopsy piece.
  In this case, the tumor size is unknown.
            """,
            "re_tumor_0":"récepteurs aux œstrogènes (RE ou RO) ('positifs', 'négatifs')",
            "rp_tumor_0":"récepteurs à la progestérone (RP) ('positifs', 'négatifs')",
            "embols_vasculaires_tumor_0":"présence de emboles vasculaires ('oui','non')",
            "cerb_tumor_0":"statut HER2 (aussi nommé CerbB-2 ou HER-2/neu)",
            "marges_saines_tumor_0":"marges saines ('oui','non')",
            "ki67_tumor_0":"index de prolifération Ki-67 (%)",
            "ref_morpho_name_tumor_0" :"diagnostic histologique ou type morphologique",
            "ref_grade_tumor_0" : "grade histopronostique SBR (I,II,III)",
            }

def call_extract_val(att,expected_value,texte,sampled_vals):
    # att_example_input=[text_ex1_by_att,text_ex2_by_att]
    # att_example_output=[ex1_by_att[att], ex2_by_att[att]]
    
    message=build_eval_val_prompt(att,
                                  dict_att_names[att],
                                  expected_value,
                                  # [ex1_by_att[att], ex2_by_att[att]], 
                                  # [text_ex1_by_att, text_ex2_by_att], 
                                  texte
                                 )
    # att,attribute_name, att_example_output, example_textes, texte)
    # print(message)
    
    response = client.chat.completions.parse(
        model=model_id,
        messages=message,
        # max_tokens=10,
        temperature=0.0,
        logprobs=True,
        top_logprobs=5,
        response_format=ExtractionS,
    )
    
    # get generated token
    reponse = response.choices[0].message
    return reponse
    

In [27]:
def evaluate_generation(df_generation): 
    count=0
    res_eval=[]
    for ind in df_generation.index: 
        print(f"***Ind:{ind}***")

        count=count+1

        # get the attributes
        entry_data=df_generation["entry_data"].loc[ind]
        texte=df_generation["generated_text"].loc[ind]
        
        extracted_attributes={}
        for att,val in entry_data.items():
            if att in ['original_index',"t","n","m","re_perc", "rp_perc"]:
                continue
            # print(att,val)
            if att == "type_diagnostique" and val == "unknown":
                val="tumorectomie"
                if (entry_data.get("taille_tumor_0","unknown")=="unknown"):
                    val="biopsie"
                entry_data[att]=val
            response= call_extract_val(att,val,texte,False)
            if getattr(response, "refusal", None):
                print(response.refusal)
                json_response = {"correct": "refusal", "reference": "-"}
            elif getattr(response, "parsed", None) is not None:
                # print(response.parsed)
                json_response = {
                    "correct": response.parsed.correct,
                    "reference": response.parsed.reference,
                }
            else:
                print("⚠️ No parsed object; raw message content:", response)
                json_response = {"correct": "unknown", "reference": "-"}
            extracted_attributes[att]=json_response
                    
        res_eval.append({
        "index": ind,
        "entry_data":entry_data,
        "text": texte,
        "extracted_attributes": extracted_attributes
        })        
                
        # print("------------\n")
    return res_eval




In [8]:
def compute_scores(res_eval):
    count_total=0
    count_correct=0
    per_doc={}
    per_attribute={'type_diagnostique': {'total': 0,
                                         'correct': 0},
       'ganglions_preleves': {'total': 0,
                              'correct': 0},
       'ganglions_atteints': {'total': 0,
                              'correct': 0},
       'rupture_capsulaire': {'total': 0,
                              'correct': 0},
       'taille_tumor_0': {'total': 0,
                          'correct': 0},
       're_tumor_0': {'total': 0,
                      'correct': 0},
       'rp_tumor_0': {'total': 0,
                      'correct': 0},
       'embols_vasculaires_tumor_0': {'total': 0,
                                      'correct': 0},
       'cerb_tumor_0': {'total': 0,
                        'correct': 0},
       'marges_saines_tumor_0': {'total': 0,
                                 'correct': 0},
       'ki67_tumor_0': {'total': 0,
                        'correct': 0},
       'ref_morpho_name_tumor_0': {'total': 0,
                                   'correct': 0},
       'ref_grade_tumor_0': {'total': 0,
                                   'correct': 0},
         }
    
    for ind,elem in enumerate(res_eval): 
        print(elem)
        count_doc=0
        correct_doc=0
        entry_data=elem["entry_data"]
        for att, val in entry_data.items(): 
            if att in ["original_index","t","n","m",'re_perc','rp_perc']:
                continue
            
            count_total+=1
            count_doc+=1
            per_attribute[att]["total"]+=1
            if val=="unknown":
                # print(elem["extracted_attributes"][att])
                if elem["extracted_attributes"][att]["correct"]=="no":
                    count_correct+=1
                    correct_doc+=1
                    per_attribute[att]["correct"]+=1
            else:
                if elem["extracted_attributes"][att]["correct"]=="yes":
                    count_correct+=1
                    correct_doc+=1
                    per_attribute[att]["correct"]+=1
        per_doc[ind]=correct_doc/count_doc
    return count_correct,count_total,per_attribute,per_doc
            

In [ ]:
def bucket_performance(per_doc):
    buckets = {
        "perfect_1_00": [],
        "high_0_85_to_0_99": [],
        "moderate_0_70_to_0_84": [],
        "low_0_50_to_0_69": [],
        "very_low_below_0_50": []
    }

    for doc_id, score in per_doc.items():
        if score == 1.0:
            buckets["perfect_1_00"].append(doc_id)
        elif 0.85 <= score < 1.0:
            buckets["high_0_85_to_0_99"].append(doc_id)
        elif 0.70 <= score < 0.85:
            buckets["moderate_0_70_to_0_84"].append(doc_id)
        elif 0.50 <= score < 0.70:
            buckets["low_0_50_to_0_69"].append(doc_id)
        else:
            buckets["very_low_below_0_50"].append(doc_id)

    return buckets




# Medgemma train evaluation

In [2]:
evalutaion={}


now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

df_generation= pd.read_json("../TextGeneration/Ouptut/outputgreedy_google_medgemma-27b-text-itTrain.json")
# Remove keys from each dictionary in the 'data' column
keys_to_remove = ["original_index","t", "n", "m"]
df_generation['entry_data'] = df_generation['entry_data'].apply(lambda x: {k: v for k, v in x.items() if k not in keys_to_remove})

print("**************************Medgemma greedy******************************")
res_eval_medgemma_greedyL=evaluate_generation(df_generation)

# save evaluation: 
output_path = f"LLMEvaluation/Eval_google_medgemma-27b-text-itTrain_{llm_used}.json"
# Save to JSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(res_eval_medgemma_greedyL, f, ensure_ascii=False, indent=4)


count_correct_gemma_greedyL,count_total_gemma_greedyL,per_attribute_gemma_greedyL,per_doc_gemma_greedyL= compute_scores(res_eval_medgemma_greedyL)
evalutaion["med_gemma_L"]={
   'eval':res_eval_medgemma_greedyL,
    'count_correct':count_correct_gemma_greedyL,
    'count_total':count_total_gemma_greedyL,
    'per_att':per_attribute_gemma_greedyL,
    'per_doc':per_doc_gemma_greedyL
}

now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

In [46]:
count_correct_gemma_greedyL,count_total_gemma_greedyL,per_attribute_gemma_greedyL,per_doc_gemma_greedyL= compute_scores(res_eval_medgemma_greedyL)
evalutaion["med_gemma_L"]={
   'eval':res_eval_medgemma_greedyL,
    'count_correct':count_correct_gemma_greedyL,
    'count_total':count_total_gemma_greedyL,
    'per_att':per_attribute_gemma_greedyL,
    'per_doc':per_doc_gemma_greedyL
}

print("--------------------med_gemma TRain------------------------------------")
print(f"count_correct: {evalutaion["med_gemma_L"]['count_correct']}" )
print(f"count_total: {evalutaion["med_gemma_L"]['count_total']}" )
print(f"per_att':: {evalutaion["med_gemma_L"]['per_att']}" )
print(f"per_doc: {evalutaion["med_gemma_L"]['per_doc']}" )

# med_gemma:
buckets = bucket_performance(evalutaion["med_gemma_L"]["per_doc"])


print()
print("BUCKETS")
for name, docs in buckets.items():
    print(f"{name}: {len(docs)} docs")
    print(docs[:20])   # preview first 20 doc IDs
    print()



--------------------med_gemma TRain------------------------------------
count_correct: 13356
count_total: 13806
per_att':: {'type_diagnostique': {'total': 1062, 'correct': 1059}, 'ganglions_preleves': {'total': 1062, 'correct': 1014}, 'ganglions_atteints': {'total': 1062, 'correct': 1060}, 'rupture_capsulaire': {'total': 1062, 'correct': 1008}, 'taille_tumor_0': {'total': 1062, 'correct': 1051}, 're_tumor_0': {'total': 1062, 'correct': 1062}, 'rp_tumor_0': {'total': 1062, 'correct': 1062}, 'embols_vasculaires_tumor_0': {'total': 1062, 'correct': 1029}, 'cerb_tumor_0': {'total': 1062, 'correct': 929}, 'marges_saines_tumor_0': {'total': 1062, 'correct': 967}, 'ki67_tumor_0': {'total': 1062, 'correct': 1062}, 'ref_morpho_name_tumor_0': {'total': 1062, 'correct': 1022}, 'ref_grade_tumor_0': {'total': 1062, 'correct': 1031}}
per_doc: {0: 0.9230769230769231, 1: 1.0, 2: 1.0, 3: 1.0, 4: 0.9230769230769231, 5: 1.0, 6: 0.9230769230769231, 7: 1.0, 8: 1.0, 9: 1.0, 10: 0.9230769230769231, 11: 1.0, 

# Medgemma RAG Train Evaluation: 

In [3]:
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

df_generation= pd.read_json("../TextGeneration/Ouptut/outputgreedy_google_medgemma-27b-text-it_RAG_k2Train.json")
# df_generation = df_generation.drop(columns=["index"])
# Remove keys from each dictionary in the 'data' column
keys_to_remove = ["original_index","t", "n", "m"]
df_generation['entry_data'] = df_generation['entry_data'].apply(lambda x: {k: v for k, v in x.items() if k not in keys_to_remove})

print("**************************Medgemma greedy RAG******************************")
res_eval_medgemma_RAG=evaluate_generation(df_generation)

# save evaluation: 
output_path = f"LLMEvaluation/Eval_google_medgemma-27b-text-it_RAG_Train_{llm_used}.json"
# Save to JSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(res_eval_medgemma_RAG, f, ensure_ascii=False, indent=4)


count_correct_gemma_greedyL,count_total_gemma_greedyL,per_attribute_gemma_greedyL,per_doc_gemma_greedyL= compute_scores(res_eval_medgemma_RAG)
evalutaion["med_gemma_RAG"]={
   'eval':res_eval_medgemma_RAG,
    'count_correct':count_correct_gemma_greedyL,
    'count_total':count_total_gemma_greedyL,
    'per_att':per_attribute_gemma_greedyL,
    'per_doc':per_doc_gemma_greedyL
}

buckets_gemmaRAG = bucket_performance(evalutaion["med_gemma_RAG"]["per_doc"])
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))



In [54]:
print("--------------------med_gemma_RAG------------------------------------")
print(f"count_correct: {evalutaion["med_gemma_RAG"]['count_correct']}" )
print(f"count_total: {evalutaion["med_gemma_RAG"]['count_total']}" )
print(f"per_att':: {evalutaion["med_gemma_RAG"]['per_att']}" )
print(f"per_doc: {evalutaion["med_gemma_RAG"]['per_doc']}" )

print()
print("BUCKETS")
for name, docs in buckets_gemmaRAG.items():
    print(f"{name}: {len(docs)} docs")
    print(docs[:20])   # preview first 20 doc IDs
    print()

    

--------------------med_gemma_RAG------------------------------------
count_correct: 13266
count_total: 13806
per_att':: {'type_diagnostique': {'total': 1062, 'correct': 1061}, 'ganglions_preleves': {'total': 1062, 'correct': 1009}, 'ganglions_atteints': {'total': 1062, 'correct': 999}, 'rupture_capsulaire': {'total': 1062, 'correct': 1006}, 'taille_tumor_0': {'total': 1062, 'correct': 1052}, 're_tumor_0': {'total': 1062, 'correct': 1062}, 'rp_tumor_0': {'total': 1062, 'correct': 1062}, 'embols_vasculaires_tumor_0': {'total': 1062, 'correct': 1004}, 'cerb_tumor_0': {'total': 1062, 'correct': 1025}, 'marges_saines_tumor_0': {'total': 1062, 'correct': 925}, 'ki67_tumor_0': {'total': 1062, 'correct': 1061}, 'ref_morpho_name_tumor_0': {'total': 1062, 'correct': 1038}, 'ref_grade_tumor_0': {'total': 1062, 'correct': 962}}
per_doc: {0: 0.9230769230769231, 1: 0.8461538461538461, 2: 0.9230769230769231, 3: 1.0, 4: 1.0, 5: 0.8461538461538461, 6: 0.8461538461538461, 7: 1.0, 8: 0.9230769230769231,

# Mixtral Train Evaluation 

In [4]:
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

df_generation= pd.read_json("../TextGeneration/Ouptut/outputgreedy_mistralai_Mixtral-8x7B-Instruct-v0.1Train.json")
# df_generation = df_generation.drop(columns=["index"])
# Remove keys from each dictionary in the 'data' column
keys_to_remove = ["original_index","t", "n", "m"]
df_generation['entry_data'] = df_generation['entry_data'].apply(lambda x: {k: v for k, v in x.items() if k not in keys_to_remove})

print("**************************Mixtral Train******************************")
res_mixtralTrain=evaluate_generation(df_generation)

# save evaluation: 
output_path = f"LLMEvaluation/Eval_Mixtral-8x7B-Instruct-v0.1Train_{llm_used}.json"
# Save to JSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(res_mixtralTrain, f, ensure_ascii=False, indent=4)


count_correct_mixtralTrain,count_total_mixtralTrain,per_attribute_mixtralTrain,per_doc_mixtralTrain= compute_scores(res_mixtralTrain)
evalutaion["mixtral_train"]={
   'eval':res_mixtralTrain,
    'count_correct':count_correct_mixtralTrain,
    'count_total':count_total_mixtralTrain,
    'per_att':per_attribute_mixtralTrain,
    'per_doc':per_doc_mixtralTrain
}

buckets_mixtral_train = bucket_performance(evalutaion["mixtral_train"]["per_doc"])
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

In [48]:
print("--------------------mixtral_train------------------------------------")
print(f"count_correct: {evalutaion["mixtral_train"]['count_correct']}" )
print(f"count_total: {evalutaion["mixtral_train"]['count_total']}" )
print(f"per_att':: {evalutaion["mixtral_train"]['per_att']}" )
print(f"per_doc: {evalutaion["mixtral_train"]['per_doc']}" )

print()
print("BUCKETS")
for name, docs in buckets_mixtral_train.items():
    print(f"{name}: {len(docs)} docs")
    print(docs[:20])   # preview first 20 doc IDs
    print()

    

--------------------mixtral_train------------------------------------
count_correct: 13025
count_total: 13806
per_att':: {'type_diagnostique': {'total': 1062, 'correct': 1062}, 'ganglions_preleves': {'total': 1062, 'correct': 942}, 'ganglions_atteints': {'total': 1062, 'correct': 950}, 'rupture_capsulaire': {'total': 1062, 'correct': 935}, 'taille_tumor_0': {'total': 1062, 'correct': 1044}, 're_tumor_0': {'total': 1062, 'correct': 1054}, 'rp_tumor_0': {'total': 1062, 'correct': 1059}, 'embols_vasculaires_tumor_0': {'total': 1062, 'correct': 920}, 'cerb_tumor_0': {'total': 1062, 'correct': 912}, 'marges_saines_tumor_0': {'total': 1062, 'correct': 998}, 'ki67_tumor_0': {'total': 1062, 'correct': 1060}, 'ref_morpho_name_tumor_0': {'total': 1062, 'correct': 1042}, 'ref_grade_tumor_0': {'total': 1062, 'correct': 1047}}
per_doc: {0: 1.0, 1: 1.0, 2: 0.9230769230769231, 3: 0.6923076923076923, 4: 0.9230769230769231, 5: 0.8461538461538461, 6: 1.0, 7: 0.9230769230769231, 8: 0.6923076923076923, 9:

In [ ]:
# ******************************************************

# Mixtral Train RAG 

In [5]:
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

df_generation= pd.read_json("../TextGeneration/Ouptut/outputgreedy_mistralai_Mixtral-8x7B-Instruct-v0.1_RAG_k2Train.json")
# df_generation = df_generation.drop(columns=["index"])
# Remove keys from each dictionary in the 'data' column
keys_to_remove = ["original_index","t", "n", "m"]
df_generation['entry_data'] = df_generation['entry_data'].apply(lambda x: {k: v for k, v in x.items() if k not in keys_to_remove})

print("**************************Mixtral Train RAG******************************")
res_mixtralTrain_RAG=evaluate_generation(df_generation)

# save evaluation: 
output_path = f"LLMEvaluation/Eval_Mixtral-8x7B-Instruct-v0.1Train_RAG_{llm_used}.json"
# Save to JSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(res_mixtralTrain_RAG, f, ensure_ascii=False, indent=4)


count_correct_mixtralTrain_RAG,count_total_mixtralTrain_RAG,per_attribute_mixtralTrain_RAG,per_doc_mixtralTrain_RAG= compute_scores(res_mixtralTrain_RAG)
evalutaion["mixtral_train_RAG"]={
   'eval':res_mixtralTrain_RAG,
    'count_correct':count_correct_mixtralTrain_RAG,
    'count_total':count_total_mixtralTrain_RAG,
    'per_att':per_attribute_mixtralTrain_RAG,
    'per_doc':per_doc_mixtralTrain_RAG
}

buckets_mixtral_train_RAG = bucket_performance(evalutaion["mixtral_train_RAG"]["per_doc"])
now = datetime.now()
print("Heure actuelle :", now.strftime("%H:%M:%S"))

In [56]:
print("--------------------mixtral_train RAG-----------------------------------")
print(f"count_correct: {evalutaion["mixtral_train_RAG"]['count_correct']}" )
print(f"count_total: {evalutaion["mixtral_train_RAG"]['count_total']}" )
print(f"per_att':: {evalutaion["mixtral_train_RAG"]['per_att']}" )
print(f"per_doc: {evalutaion["mixtral_train_RAG"]['per_doc']}" )

print()
print("BUCKETS")
for name, docs in buckets_mixtral_train_RAG.items():
    print(f"{name}: {len(docs)} docs")
    print(docs[:20])   # preview first 20 doc IDs
    print()


--------------------mixtral_train RAG-----------------------------------
count_correct: 12550
count_total: 13806
per_att':: {'type_diagnostique': {'total': 1062, 'correct': 1060}, 'ganglions_preleves': {'total': 1062, 'correct': 967}, 'ganglions_atteints': {'total': 1062, 'correct': 918}, 'rupture_capsulaire': {'total': 1062, 'correct': 763}, 'taille_tumor_0': {'total': 1062, 'correct': 994}, 're_tumor_0': {'total': 1062, 'correct': 1050}, 'rp_tumor_0': {'total': 1062, 'correct': 1049}, 'embols_vasculaires_tumor_0': {'total': 1062, 'correct': 739}, 'cerb_tumor_0': {'total': 1062, 'correct': 965}, 'marges_saines_tumor_0': {'total': 1062, 'correct': 928}, 'ki67_tumor_0': {'total': 1062, 'correct': 1052}, 'ref_morpho_name_tumor_0': {'total': 1062, 'correct': 1031}, 'ref_grade_tumor_0': {'total': 1062, 'correct': 1034}}
per_doc: {0: 0.8461538461538461, 1: 0.8461538461538461, 2: 0.8461538461538461, 3: 0.7692307692307693, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 0.9230769230769231, 9: 1.0, 10: 1.0

# Comparisons

In [6]:
eval_att_medgemma = pd.read_json("LLMEvaluation/0Eval_google_medgemma-27b-text-itTrain_gpt-5.1.json")
eval_att_mixtral = pd.read_json("LLMEvaluation/0Eval_Mixtral-8x7B-Instruct-v0.1Train_gpt-5.1.json")

eval_att_medgemmaRAG = pd.read_json("LLMEvaluation/0Eval_google_medgemma-27b-text-it_RAG_Train_gpt-5.1.json")
eval_att_mixtralRAG = pd.read_json("LLMEvaluation/0Eval_Mixtral-8x7B-Instruct-v0.1Train_RAG_gpt-5.1.json")
eval_att_medgemma.head(1).to_dict()

In [7]:

evalutaion={}
count_correct_gemma_greedyL,count_total_gemma_greedyL,per_attribute_gemma_greedyL,per_doc_gemma_greedyL= compute_scores(eval_att_medgemma.to_dict(orient="records"))
evalutaion["med_gemma_L"]={
   'eval':eval_att_medgemma.to_dict(orient="records"),
    'count_correct':count_correct_gemma_greedyL,
    'count_total':count_total_gemma_greedyL,
    'per_att':per_attribute_gemma_greedyL,
    'per_doc':per_doc_gemma_greedyL
}
count_correct_gemma_greedyL,count_total_gemma_greedyL,per_attribute_gemma_greedyL,per_doc_gemma_greedyL= compute_scores(eval_att_medgemmaRAG.to_dict(orient="records"))
evalutaion["med_gemma_RAG"]={
   'eval':eval_att_medgemmaRAG.to_dict(orient="records"),
    'count_correct':count_correct_gemma_greedyL,
    'count_total':count_total_gemma_greedyL,
    'per_att':per_attribute_gemma_greedyL,
    'per_doc':per_doc_gemma_greedyL
}


count_correct_mixtralTrain,count_total_mixtralTrain,per_attribute_mixtralTrain,per_doc_mixtralTrain= compute_scores(eval_att_mixtral.to_dict(orient="records"))
evalutaion["mixtral_train"]={
   'eval':eval_att_mixtral.to_dict(orient="records"),
    'count_correct':count_correct_mixtralTrain,
    'count_total':count_total_mixtralTrain,
    'per_att':per_attribute_mixtralTrain,
    'per_doc':per_doc_mixtralTrain
}


count_correct_mixtralTrain_RAG,count_total_mixtralTrain_RAG,per_attribute_mixtralTrain_RAG,per_doc_mixtralTrain_RAG= compute_scores(eval_att_mixtralRAG.to_dict(orient="records"))
evalutaion["mixtral_train_RAG"]={
   'eval':eval_att_mixtralRAG.to_dict(orient="records"),
    'count_correct':count_correct_mixtralTrain_RAG,
    'count_total':count_total_mixtralTrain_RAG,
    'per_att':per_attribute_mixtralTrain_RAG,
    'per_doc':per_doc_mixtralTrain_RAG
}

In [21]:
# compute % correct per attribute/model
all_models_per_att={
    "medGemma": evalutaion["med_gemma_L"]['per_att'],
    "medGemmaRAG": evalutaion["med_gemma_RAG"]['per_att'],
    "mixtral": evalutaion["mixtral_train"]['per_att'],
    "mixtralRAG": evalutaion["mixtral_train_RAG"]['per_att'],
}
rows = {}

for model_name, per_att in all_models_per_att.items():
    for attribute, scores in per_att.items():
        percentage = scores["correct"] / scores["total"] * 100
        rows.setdefault(attribute, {})[model_name] = percentage

df_percent = pd.DataFrame.from_dict(rows, orient="index")

# Optional formatting
df_percent = df_percent.round(2)

df_percent

,medGemma,medGemmaRAG,mixtral,mixtralRAG
type_diagnostique,99.72,99.91,100.00,99.81
ganglions_preleves,95.48,95.01,88.70,91.05
ganglions_atteints,99.81,94.07,89.45,86.44
rupture_capsulaire,94.92,94.73,88.04,71.85
taille_tumor_0,98.96,99.06,98.31,93.60
re_tumor_0,100.00,100.00,99.25,98.87
rp_tumor_0,100.00,100.00,99.72,98.78
embols_vasculaires_tumor_0,96.89,94.54,86.63,69.59
cerb_tumor_0,87.48,96.52,85.88,90.87
marges_saines_tumor_0,91.05,87.10,93.97,87.38


# confussion matrix

In [34]:
def is_attribute_correct(attr_data):
    val = attr_data.get("value")

    label_correct = attr_data.get("correct")

    if val == "unknown":
        return label_correct == "no"

    return label_correct == "yes"


def extract_correctness(df, attributes):
    correctness = pd.DataFrame(index=df.index)

    for attr in attributes:
        correctness[attr] = df["extracted_attributes"].apply(
            lambda x: is_attribute_correct(x.get(attr, {}))
        )

    return correctness

In [35]:
import pandas as pd
import numpy as np

models = {
    "medGemma": eval_att_medgemma,
    "medGemmaRAG": eval_att_medgemmaRAG,
    "mixtral": eval_att_mixtral,
    "mixtralRAG": eval_att_mixtralRAG,
}

attributes = [
    "type_diagnostique",
    "ganglions_preleves",
    "ganglions_atteints",
    "rupture_capsulaire",
    "taille_tumor_0",
    "re_tumor_0",
    "rp_tumor_0",
    "embols_vasculaires_tumor_0",
    "cerb_tumor_0",
    "marges_saines_tumor_0",
    "ki67_tumor_0",
    "ref_morpho_name_tumor_0",
    "ref_grade_tumor_0",
]

In [37]:
def compute_pairwise_error_matrix(correctness_by_model, attr):
    model_names = list(correctness_by_model.keys())
    matrix = pd.DataFrame(index=model_names, columns=model_names, dtype=float)

    n_docs = len(next(iter(correctness_by_model.values())))

    for row_model in model_names:
        row_correct = correctness_by_model[row_model][attr]

        for col_model in model_names:
            col_correct = correctness_by_model[col_model][attr]

            percentage = ((row_correct) & (~col_correct)).sum() / n_docs * 100
            matrix.loc[row_model, col_model] = percentage

    return matrix.round(2)

correctness_by_model = {
    model_name: extract_correctness(df, attributes)
    for model_name, df in models.items()
}

confusion_matrices = {
    attr: compute_pairwise_error_matrix(correctness_by_model, attr)
    for attr in attributes
}

In [39]:
def latex_escape(text):
    return text.replace("_", r"\_")


def confusion_matrix_to_latex(attr, matrix, caption_prefix="Pairwise confusion matrix"):
    latex = []

    label_attr = attr.replace("_", "-")

    latex.append(r"\begin{table}[ht]")
    latex.append(r"\centering")
    latex.append(
        rf"\caption{{{caption_prefix} for attribute \texttt{{{latex_escape(attr)}}}. "
        r"Each cell reports the percentage of documents where the row model is correct "
        r"and the column model is incorrect.}}"
    )
    latex.append(rf"\label{{tab:confusion-{label_attr}}}")

    col_format = "l" + "r" * len(matrix.columns)
    latex.append(rf"\begin{{tabular}}{{{col_format}}}")
    latex.append(r"\hline")

    header = " & ".join(
        [r"\textbf{Model}"] + [rf"\textbf{{{latex_escape(col)}}}" for col in matrix.columns]
    )
    latex.append(header + r" \\")
    latex.append(r"\hline")

    for row_name, row in matrix.iterrows():
        values = []
        for col_name, value in row.items():
            values.append(f"{value:.2f}")

        latex.append(
            latex_escape(row_name) + " & " + " & ".join(values) + r" \\"
        )

    latex.append(r"\hline")
    latex.append(r"\end{tabular}")
    latex.append(r"\end{table}")

    return "\n".join(latex)

In [40]:
latex_tables = {
    attr: confusion_matrix_to_latex(attr, matrix)
    for attr, matrix in confusion_matrices.items()
}

In [205]:
total_doc=len(coherence_eval)
correct_doc=0
for ind,elem in enumerate(coherence_eval): 
    if elem['coherence_evaluation']=='PASS':
        correct_doc+=1
print(f"coherence_eval: {correct_doc/total_doc}")


coherence_eval: 0.75
